In [1]:
import scdrs
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import scipy.io

print("scDRS version:", scdrs.__version__)
print("scanpy version:", sc.__version__)

scDRS version: 1.0.2
scanpy version: 1.12.3


/var/folders/k0/d72_1mdx5kj_sd9dvg_s39cr0000gn/T/ipykernel_76550/1315736049.py:9: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print("scanpy version:", sc.__version__)


In [2]:
counts = scipy.io.mmread("/Users/lexigover/Desktop/OSD-404/GLDS404_STARsolo/counts.mtx").T.tocsr()
genes = pd.read_csv("/Users/lexigover/Desktop/OSD-404/GLDS404_STARsolo/genes.csv")["x"].values
meta = pd.read_csv("/Users/lexigover/Desktop/OSD-404/GLDS404_STARsolo/metadata.csv", index_col=0)

/var/folders/k0/d72_1mdx5kj_sd9dvg_s39cr0000gn/T/ipykernel_76550/3857002223.py:1: DeprecationWarning: The default value for `spmatrix` is changing to `False` in v1.20.
             That means the default return type will be a sparse array.
             Unless you use * instead of @, ** for matrix power, or you depend
             on 2D shapes from e.g. `A.sum(axis=0)` it may not matter to you.
             See the spmatrix to sparray migration guide for details.
             https://docs.scipy.org/doc/scipy/reference/sparse.migration_to_sparray.html
             
  counts = scipy.io.mmread("/Users/lexigover/Desktop/OSD-404/GLDS404_STARsolo/counts.mtx").T.tocsr()


In [3]:
print("counts shape:", counts.shape)
print("genes:", len(genes))
print("metadata shape:", meta.shape)

counts shape: (25294, 19649)
genes: 19649
metadata shape: (25294, 27)


In [4]:
adata = ad.AnnData(X=counts, obs=meta, var=pd.DataFrame(index=genes))
adata.var_names = genes
adata

AnnData object with n_obs × n_vars = 25294 × 19649
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'condition', 'age', 'percent.mt', 'percent.ribo', 'RNA_snn_res.0.5', 'seurat_clusters', 'RNA_snn_res.0.3', 'RNA_snn_res.0.8', 'RNA_snn_res.1', 'singler_labels', 'singler_labels_clean', 'cell_type', 'scDblFinder_class', 'MitoCarta_score1', 'Mitophagy_score1', 'cGAS_STING_score1', 'NLRP3_score1', 'AIM2_score1', 'IFN_score1', 'percent.hb', 'cell_type_cc', 'celltype_fine', 'celltype_broad'
    layers: None (.X)

In [5]:
# confirm cell_type and condition made it over correctly
adata.obs['cell_type'].value_counts()
adata.obs['condition'].value_counts()

# confirm counts are raw integers (scDRS expects raw counts, not normalized data)
adata.X[:5,:5].toarray()  # should show whole numbers, not decimals

array([[0, 0, 0, 1, 0],
       [2, 0, 0, 0, 0],
       [1, 0, 0, 0, 0],
       [1, 0, 0, 0, 0],
       [0, 0, 0, 0, 0]])

In [6]:
adata.write_h5ad("combined_for_scdrs.h5ad")

In [7]:
adata.obs['cell_type'].value_counts()
adata.obs['condition'].value_counts()

condition
GC     14758
FLT    10536
Name: count, dtype: int64

In [8]:
!ls /Users/lexigover/Desktop/OSD-404/GLDS404_STARsolo/single_cell_data/zeisel_2015

cov.tsv    expr.h5ad  geneset.gs


In [9]:
import pandas as pd

gs_path = "/Users/lexigover/Desktop/OSD-404/GLDS404_STARsolo/single_cell_data/zeisel_2015/geneset.gs"
df_gs = pd.read_csv(gs_path, sep="\t", index_col=0)

df_gs.index.tolist()  

['PASS_ADHD_Demontis2018',
 'PASS_Alzheimers_Jansen2019',
 'PASS_BIP_Mullins2021',
 'PASS_BIP_Stahl2019',
 'PASS_DrinksPerWeek_Liu2019',
 'PASS_GeneralRiskTolerance_KarlssonLinner2019',
 'PASS_Insomnia_Jansen2019',
 'PASS_Intelligence_SavageJansen2018',
 'PASS_MDD_Howard2019',
 'PASS_ReactionTime_Davies2018',
 'PASS_SWB',
 'PASS_Schizophrenia_Pardinas2018',
 'PASS_SleepDuration_Dashti2019',
 'PASS_VerbalNumericReasoning_Davies2018',
 'PASS_Worry_Nagel2018',
 'UKB_460K.body_BMIz',
 'UKB_460K.body_HEIGHTz',
 'UKB_460K.cov_EDU_COLLEGE',
 'UKB_460K.cov_EDU_YEARS',
 'UKB_460K.cov_SMOKING_STATUS',
 'UKB_460K.mental_NEUROTICISM',
 'UKB_460K.other_MORNINGPERSON',
 'UKB_460K.repro_NumberChildrenEverBorn_Pooled',
 'spatial_ventral',
 'spatial_dorsal',
 'spatial_distal',
 'spatial_proximal',
 'spatial_deep',
 'spatial_superficial']

In [10]:
adata.write_h5ad("/Users/lexigover/Desktop/OSD-404/GLDS404_STARsolo/combined_for_scdrs.h5ad")

In [11]:
import os
size = os.path.getsize("/Users/lexigover/Desktop/OSD-404/GLDS404_STARsolo/combined_for_scdrs.h5ad")
print(f"{size / 1e6:.1f} MB")

251.9 MB


In [12]:
import scanpy as sc
adata_check = sc.read_h5ad("/Users/lexigover/Desktop/OSD-404/GLDS404_STARsolo/combined_for_scdrs.h5ad")
adata_check

AnnData object with n_obs × n_vars = 25294 × 19649
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'condition', 'age', 'percent.mt', 'percent.ribo', 'RNA_snn_res.0.5', 'seurat_clusters', 'RNA_snn_res.0.3', 'RNA_snn_res.0.8', 'RNA_snn_res.1', 'singler_labels', 'singler_labels_clean', 'cell_type', 'scDblFinder_class', 'MitoCarta_score1', 'Mitophagy_score1', 'cGAS_STING_score1', 'NLRP3_score1', 'AIM2_score1', 'IFN_score1', 'percent.hb', 'cell_type_cc', 'celltype_fine', 'celltype_broad'
    layers: None (.X)

In [13]:
import pandas as pd
import scdrs

GS_FILE = "/Users/lexigover/Desktop/OSD-404/GLDS404_STARsolo/single_cell_data/zeisel_2015/geneset.gs"
df_gs = scdrs.util.load_gs(GS_FILE)

gene_list = df_gs['PASS_Alzheimers_Jansen2019'][0]
gene_weight = df_gs['PASS_Alzheimers_Jansen2019'][1]

print(len(gene_list))
print(gene_list[:10])

845
['Bcam', 'Ceacam16', 'Exoc3l2', 'Ms4a6d', 'Clu', 'Cblc', 'Ms4a2', 'Picalm', 'Clptm1', 'Slc24a4']


In [14]:
!ls /Users/lexigover/Desktop/OSD-404/GLDS404_STARsolo/*gs_file

magma_10kb_top1000_zscore.74_traits.rv1.gs
magma_10kb_top1000_zscore.75_traits.batch


In [15]:
import pandas as pd

GS_FILE_FULL = "/Users/lexigover/Desktop/OSD-404/GLDS404_STARsolo/gs_file/magma_10kb_top1000_zscore.74_traits.rv1.gs"
df_gs_full = pd.read_csv(GS_FILE_FULL, sep="\t", index_col=0)

pd.set_option('display.max_rows', None)
df_gs_full.index.tolist()

['PASS_ADHD_Demontis2018',
 'PASS_Alzheimers_Jansen2019',
 'PASS_AtrialFibrillation_Nielsen2018',
 'PASS_BIP_Mullins2021',
 'PASS_CD_deLange2017',
 'PASS_Celiac',
 'PASS_Coronary_Artery_Disease',
 'PASS_DrinksPerWeek_Liu2019',
 'PASS_FastingGlucose_Manning',
 'PASS_GeneralRiskTolerance_KarlssonLinner2019',
 'PASS_IBD_deLange2017',
 'PASS_Insomnia_Jansen2019',
 'PASS_Intelligence_SavageJansen2018',
 'PASS_Lupus',
 'PASS_MDD_Howard2019',
 'PASS_Multiple_sclerosis',
 'PASS_Primary_biliary_cirrhosis',
 'PASS_ReactionTime_Davies2018',
 'PASS_Rheumatoid_Arthritis',
 'PASS_SWB',
 'PASS_Schizophrenia_Pardinas2018',
 'PASS_SleepDuration_Dashti2019',
 'PASS_Type_1_Diabetes',
 'PASS_Type_2_Diabetes',
 'PASS_UC_deLange2017',
 'PASS_VerbalNumericReasoning_Davies2018',
 'PASS_Worry_Nagel2018',
 'UKB_460K.biochemistry_AlanineAminotransferase',
 'UKB_460K.biochemistry_AlkalinePhosphatase',
 'UKB_460K.biochemistry_Cholesterol',
 'UKB_460K.biochemistry_Glucose',
 'UKB_460K.biochemistry_HDLcholesterol',


In [16]:
traits_to_test = {
    "Alzheimers": "PASS_Alzheimers_Jansen2019",
    "Autoimmune_disease": "UKB_460K.disease_AID_ALL", 
    "Multiple_sclerosis": "PASS_Multiple_sclerosis"
}

In [17]:
import scanpy as sc
import pandas as pd
import scdrs

adata = sc.read_h5ad("/Users/lexigover/Desktop/OSD-404/GLDS404_STARsolo/combined_for_scdrs.h5ad")
print(adata)

#preprocess - no covariates for this first pass
scdrs.preprocess(adata)

AnnData object with n_obs × n_vars = 25294 × 19649
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'condition', 'age', 'percent.mt', 'percent.ribo', 'RNA_snn_res.0.5', 'seurat_clusters', 'RNA_snn_res.0.3', 'RNA_snn_res.0.8', 'RNA_snn_res.1', 'singler_labels', 'singler_labels_clean', 'cell_type', 'scDblFinder_class', 'MitoCarta_score1', 'Mitophagy_score1', 'cGAS_STING_score1', 'NLRP3_score1', 'AIM2_score1', 'IFN_score1', 'percent.hb', 'cell_type_cc', 'celltype_fine', 'celltype_broad'
    layers: None (.X)


/Users/lexigover/scdrs_env/lib/python3.14/site-packages/scipy/sparse/_data.py:150: RuntimeWarning: overflow encountered in expm1
  result = op(self._deduped_data())
/Users/lexigover/scdrs_env/lib/python3.14/site-packages/scipy/sparse/_data.py:134: RuntimeWarning: overflow encountered in square
  return self._with_data(data ** n)
/Users/lexigover/scdrs_env/lib/python3.14/site-packages/scdrs/pp.py:491: RuntimeWarning: overflow encountered in square
  v_var = v_var - v_mean ** 2
/Users/lexigover/scdrs_env/lib/python3.14/site-packages/scdrs/pp.py:491: RuntimeWarning: invalid value encountered in subtract
  v_var = v_var - v_mean ** 2


In [18]:
df_gs_full = scdrs.util.load_gs(GS_FILE_FULL)

for label, trait in traits_to_test.items():
    genes = df_gs_full[trait][0]
    weights = df_gs_full[trait][1]
    print(f"{label} ({trait}): {len(genes)} genes")

Alzheimers (PASS_Alzheimers_Jansen2019): 1000 genes
Autoimmune_disease (UKB_460K.disease_AID_ALL): 1000 genes
Multiple_sclerosis (PASS_Multiple_sclerosis): 1000 genes


In [19]:
gene_list = df_gs_full["Alzheimers"] if "Alzheimers" in df_gs_full else df_gs_full[traits_to_test["Alzheimers"]][0]
print(gene_list[:10])
print(adata.var_names[:10])

['BCAM', 'CEACAM16', 'PVR', 'EXOC3L2', 'MS4A6A', 'CLU', 'CBLC', 'MS4A2', 'PICALM', 'CLPTM1']
Index(['Pcmtd1', 'Gm26901', 'Exo1', 'Uxs1', 'Dsel', 'Gmppa', 'Sntg1',
       'ENSMUSG00002075967', 'Rrs1', 'Tsn'],
      dtype='str')


In [20]:
help(scdrs.util.load_gs)

df_gs_full = scdrs.util.load_gs(
    GS_FILE_FULL,
    src_species="human",
    dst_species="mouse",
    to_intersect=adata.var_names,
)

#recheck gene counts after conversion
for label, trait in traits_to_test.items():
    genes = df_gs_full[trait][0]
    print(f"{label} ({trait}): {len(genes)} genes after mouse conversion")

Help on function load_gs in module scdrs.util:

load_gs(
    gs_path: str,
    src_species: str = None,
    dst_species: str = None,
    to_intersect: List[str] = None
) -> dict
    Load the gene set file (.gs file).

    Parameters
    ----------
    gs_path : str
        Path to the gene set file with two columns 'TRAIT' and 'GENESET', separated by tab.
        'TRAIT' column contain trait names. 'GENESET' column contain gene names (matching
        expression matrix) and gene weights (for weighted gene set). For unweighted gene set,
        the 'GENESET' column contains only gene names separated by comma, e.g.,
        "<gene1>,<gene2>,<gene3>". For weighted gene set, the 'GENESET' column contains
        gene names and weights, e.g., "<gene1>:<weight1>,<gene2>:<weight2>,<gene3>:<weight3>".
    src_species : str, default=None
        Source species, must be either 'mmusculus' or 'hsapiens' if not None
    dst_species : str, default=None
        Destination species, must be either 'm

In [21]:
results = {}

for label, trait in traits_to_test.items():
    print(f"Scoring {label}...")
    gene_list = df_gs_full[trait][0]
    gene_weight = df_gs_full[trait][1]
    
    df_res = scdrs.score_cell(adata, gene_list, gene_weight=gene_weight, n_ctrl=100)
    results[label] = df_res
    print(f"{label} done — {df_res.shape[0]} cells scored\n")

Scoring Alzheimers...


Computing control scores: 100%|██████████████| 100/100 [00:03<00:00, 32.46it/s]


Alzheimers done — 25294 cells scored

Scoring Autoimmune_disease...


Computing control scores: 100%|██████████████| 100/100 [00:03<00:00, 30.53it/s]


Autoimmune_disease done — 25294 cells scored

Scoring Multiple_sclerosis...


Computing control scores: 100%|██████████████| 100/100 [00:03<00:00, 30.78it/s]


Multiple_sclerosis done — 25294 cells scored



In [22]:
print(results["Autoimmune_disease"].isna().sum())
print(results["Autoimmune_disease"].head())

print(results["Alzheimers"]["norm_score"].describe())
print(results["Alzheimers"]["norm_score"].value_counts().head(10))

raw_score      0
norm_score     0
mc_pval        0
pval           0
nlog10_pval    0
zscore         0
dtype: int64
                                  raw_score  norm_score   mc_pval      pval  \
FLT_LAR_OLD_FO4_AAACCCAAGCCTTTGA   0.089609    1.253105  0.306931  0.087156   
FLT_LAR_OLD_FO4_AAACCCACACATCCCT   0.052976    0.662617  0.326733  0.166246   
FLT_LAR_OLD_FO4_AAACCCACATTCTCTA   0.050532    0.214455  0.366337  0.429120   
FLT_LAR_OLD_FO4_AAACCCAGTCCAAGAG   0.030705    0.486573  0.207921  0.270832   
FLT_LAR_OLD_FO4_AAACCCAGTCGTCGGT   0.018323   -1.683031  0.970297  0.986150   

                                  nlog10_pval    zscore  
FLT_LAR_OLD_FO4_AAACCCAAGCCTTTGA     1.059704  1.358479  
FLT_LAR_OLD_FO4_AAACCCACACATCCCT     0.779248  0.969105  
FLT_LAR_OLD_FO4_AAACCCACATTCTCTA     0.367421  0.178615  
FLT_LAR_OLD_FO4_AAACCCAGTCCAAGAG     0.567300  0.610298  
FLT_LAR_OLD_FO4_AAACCCAGTCGTCGGT     0.006057 -2.201511  
count    2.529400e+04
mean     9.652116e-09
std      9.424957e

In [23]:
for label in traits_to_test:
    print(f"\n=== {label} ===")
    print(results[label]["norm_score"].describe())
    print(results[label]["norm_score"].value_counts().head(5))


=== Alzheimers ===
count    2.529400e+04
mean     9.652116e-09
std      9.424957e-01
min     -1.998284e+00
25%     -8.381448e-01
50%      4.833289e-01
75%      7.427343e-01
max      4.558812e+00
Name: norm_score, dtype: float64
norm_score
 0.809878    2
 0.695861    2
-1.620223    2
 0.643876    2
 0.609931    2
Name: count, dtype: int64

=== Autoimmune_disease ===
count    2.529400e+04
mean     9.652116e-09
std      1.329525e+00
min     -5.066656e+00
25%     -9.248203e-01
50%     -3.547384e-01
75%      8.525946e-01
max      1.080852e+01
Name: norm_score, dtype: float64
norm_score
-0.909591    2
-0.317352    2
-1.390277    2
-1.090247    2
-1.094887    2
Name: count, dtype: int64

=== Multiple_sclerosis ===
count    2.529400e+04
mean    -1.206515e-09
std      5.569806e-01
min     -1.511191e+00
25%     -4.823943e-01
50%      2.200171e-01
75%      4.144746e-01
max      4.268769e+00
Name: norm_score, dtype: float64
norm_score
 0.134690    2
-0.788069    2
 0.393426    2
 0.272062    2
 0

In [24]:
for label in traits_to_test:
    adata.obs[f"{label}_norm_score"] = results[label]["norm_score"].values
    adata.obs[f"{label}_pval"] = results[label]["pval"].values

adata.obs[["cell_type", "condition", 
           "Alzheimers_norm_score", "Autoimmune_disease_norm_score", "Multiple_sclerosis_norm_score"]].head()

,cell_type,condition,Alzheimers_norm_score,Autoimmune_disease_norm_score,Multiple_sclerosis_norm_score
FLT_LAR_OLD_FO4_AAACCCAAGCCTTTGA,Monocytes (patrolling),FLT,-1.452217,1.253105,-0.839820
FLT_LAR_OLD_FO4_AAACCCACACATCCCT,B cells (naive),FLT,-1.557001,0.662617,-0.539263
FLT_LAR_OLD_FO4_AAACCCACATTCTCTA,B cells (naive),FLT,-1.740321,0.214455,-0.737957
FLT_LAR_OLD_FO4_AAACCCAGTCCAAGAG,Monocytes,FLT,0.106557,0.486573,0.059986
FLT_LAR_OLD_FO4_AAACCCAGTCGTCGGT,Monocytes,FLT,0.700610,-1.683031,0.062628


In [25]:
import scipy.stats as stats

for label in traits_to_test:
    print(f"\n=== {label}: FLT vs GC by cell type ===")
    score_col = f"{label}_norm_score"
    
    results_list = []
    for ct in adata.obs["cell_type"].unique():
        sub = adata.obs[adata.obs["cell_type"] == ct]
        flt_scores = sub[sub["condition"] == "FLT"][score_col]
        gc_scores = sub[sub["condition"] == "GC"][score_col]
        
        if len(flt_scores) > 0 and len(gc_scores) > 0:
            stat, pval = stats.mannwhitneyu(flt_scores, gc_scores)
            results_list.append({
                "cell_type": ct,
                "FLT_mean": flt_scores.mean(),
                "GC_mean": gc_scores.mean(),
                "p_value": pval
            })
    
    df_summary = pd.DataFrame(results_list)
    #BH correction, matching your R workflow
    from statsmodels.stats.multitest import multipletests
    df_summary["p_adj"] = multipletests(df_summary["p_value"], method="fdr_bh")[1]
    df_summary = df_summary.sort_values("p_adj")
    print(df_summary.to_string())


=== Alzheimers: FLT vs GC by cell type ===
                       cell_type  FLT_mean   GC_mean       p_value         p_adj
1                B cells (naive) -0.300808 -0.634119  2.073616e-14  4.561956e-13
19                   Macrophages -0.101688 -0.648349  2.512629e-11  2.763892e-10
6      B cells (atypical/memory) -1.112030 -0.962304  5.007368e-04  3.672070e-03
5                        B cells  0.513212  0.534849  9.263644e-03  5.095004e-02
4               HSCs/Progenitors  0.022515 -0.046465  1.519957e-02  6.687811e-02
14                T cells (CD8+) -0.321146 -0.389491  2.387189e-02  8.753026e-02
3                       NK cells -0.203643 -0.030414  5.711897e-02  1.795168e-01
0         Monocytes (patrolling) -0.494863 -0.383805  7.479243e-02  2.056792e-01
10               Plasmacytoid DC  0.439021  0.410017  1.070101e-01  2.615802e-01
15                  Plasma cells  0.119321  0.055541  1.253136e-01  2.756899e-01
12      Monocytes (inflammatory)  0.480381  0.445679  1.732955e-0

In [26]:
adata.obs.groupby("condition")["nCount_RNA"].median()

condition
FLT    1336.0
GC     1265.0
Name: nCount_RNA, dtype: float64

In [27]:
adata.obs["percent.hb"].describe()
adata.obs.groupby("condition")["percent.hb"].median()

condition
FLT    1.169879
GC     1.059136
Name: percent.hb, dtype: float64

In [28]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

mac_df = adata.obs[adata.obs["cell_type"] == "Macrophages"].copy()
mac_df["condition_GC"] = (mac_df["condition"] == "GC").astype(int)

model = smf.glm("Multiple_sclerosis_norm_score ~ condition_GC + Q('percent.hb') + Q('nCount_RNA')", 
                 data=mac_df).fit()
print(model.summary())

model_ad = smf.glm("Alzheimers_norm_score ~ condition_GC + Q('percent.hb') + Q('nCount_RNA')", 
                    data=mac_df).fit()
print(model_ad.summary())

model_auto = smf.glm("Autoimmune_disease_norm_score ~ condition_GC + Q('percent.hb') + Q('nCount_RNA')", 
                      data=mac_df).fit()
print(model_auto.summary())

                       Generalized Linear Model Regression Results                       
Dep. Variable:     Multiple_sclerosis_norm_score   No. Observations:                  545
Model:                                       GLM   Df Residuals:                      541
Model Family:                           Gaussian   Df Model:                            3
Link Function:                          Identity   Scale:                         0.94505
Method:                                     IRLS   Log-Likelihood:                -755.91
Date:                           Sun, 20 Sep 2026   Deviance:                       511.27
Time:                                   20:01:35   Pearson chi2:                     511.
No. Iterations:                                3   Pseudo R-squ. (CS):             0.1246
Covariance Type:                       nonrobust                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
----------------

In [29]:
"""
Convert FUMA/MAGMA gene-level output (magma.genes.out) into a scDRS .gs
gene set file for Parkinson's disease (Nalls et al. 2019 GWAS).

Mirrors the format of scDRS's pre-built 74-trait .gs file, so it can be
loaded the same way as your AD / autoimmune / MS traits:

    df_gs = scdrs.util.load_gs(
        "pd_scdrs.gs",
        src_species="human",
        dst_species="mouse",
        to_intersect=adata.var_names,
    )
"""

import pandas as pd

#Load MAGMA gene-level output
#Typical FUMA magma.genes.out columns: GENE, CHR, START, STOP, NSNPS,
#NPARAM, N, ZSTAT, P (GENE is an Entrez ID, not a symbol).
magma = pd.read_csv("/Users/lexigover/Downloads/FUMA_job768256/magma.genes.out", sep=r"\s+")
print(f"Loaded {len(magma)} genes from magma.genes.out")
print(magma.head())

Loaded 19285 genes from magma.genes.out
              GENE  CHR   START    STOP  NSNPS  NPARAM       N     ZSTAT  \
0  ENSG00000269831    1  728532  749137     34       8  482730 -1.433000   
1  ENSG00000269308    1  808043  829983     47      23  482730 -1.420200   
2  ENSG00000187634    1  850260  889955    289      48  482730 -0.285740   
3  ENSG00000268179    1  851264  876445    185      40  482730  0.049666   
4  ENSG00000188976    1  869584  904689    234      48  482730 -0.858310   

         P      SYMBOL  
0  0.92406  AL669831.1  
1  0.92223  AL645608.2  
2  0.61246      SAMD11  
3  0.48019  AL645608.1  
4  0.80464       NOC2L  


In [30]:
n_genes = len(magma)
bonferroni_p = 0.05 / n_genes
n_sig = (magma["P"] < bonferroni_p).sum()
print(f"{n_sig} genes pass Bonferroni threshold (p < {bonferroni_p:.2e})")

71 genes pass Bonferroni threshold (p < 2.59e-06)


In [31]:
known_pd_genes = ["SNCA", "GBA", "LRRK2", "MAPT", "PARK7", "PINK1", "VPS13C", "GCH1"]
found = [g for g in known_pd_genes if g in magma["SYMBOL"].values]
missing = [g for g in known_pd_genes if g not in magma["SYMBOL"].values]
print(f"Found: {found}")
print(f"Missing: {missing}")

Found: ['SNCA', 'GBA', 'LRRK2', 'MAPT', 'PARK7', 'PINK1', 'VPS13C', 'GCH1']
Missing: []


In [32]:
magma_clean = magma.dropna(subset=["SYMBOL"])
magma_sorted = magma_clean.reindex(magma_clean["ZSTAT"].abs().sort_values(ascending=False).index)
top_genes = magma_sorted.head(1000)

geneset_str = ",".join(f"{row.SYMBOL}:{row.ZSTAT:.4f}" for row in top_genes.itertuples())

with open("pd_scdrs.gs", "w") as f:
    f.write("TRAIT\tGENESET\n")
    f.write(f"PD\t{geneset_str}\n")

import os
print(f"Wrote pd_scdrs.gs with {len(top_genes)} weighted genes, file size: {os.path.getsize('pd_scdrs.gs')} bytes")

Wrote pd_scdrs.gs with 1000 weighted genes, file size: 14148 bytes


In [33]:
import scdrs

df_gs_pd = scdrs.util.load_gs(
    "pd_scdrs.gs",
    src_species="human",
    dst_species="mouse",
    to_intersect=adata.var_names,
)
gene_list = df_gs_pd["PD"][0]
gene_weight = df_gs_pd["PD"][1]
print(f"{len(gene_list)} genes after mouse conversion")

df_res_pd = scdrs.score_cell(adata, gene_list, gene_weight=gene_weight, n_ctrl=100)
results["PD"] = df_res_pd
print(f"PD done — {df_res_pd.shape[0]} cells scored")

673 genes after mouse conversion


Computing control scores: 100%|██████████████| 100/100 [00:02<00:00, 35.15it/s]


PD done — 25294 cells scored


In [34]:
results["PD"]["norm_score"].describe()
results["PD"]["norm_score"].value_counts().head(5)

norm_score
-0.198637    2
 0.874589    2
 0.877897    2
 0.449632    2
 0.614796    2
Name: count, dtype: int64

In [35]:
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests

adata.obs["PD_norm_score"] = results["PD"]["norm_score"].values
adata.obs["PD_pval"] = results["PD"]["pval"].values

score_col = "PD_norm_score"
results_list = []
for ct in adata.obs["cell_type"].unique():
    sub = adata.obs[adata.obs["cell_type"] == ct]
    flt_scores = sub[sub["condition"] == "FLT"][score_col]
    gc_scores = sub[sub["condition"] == "GC"][score_col]
    if len(flt_scores) > 0 and len(gc_scores) > 0:
        stat, pval = stats.mannwhitneyu(flt_scores, gc_scores)
        results_list.append({"cell_type": ct, "FLT_mean": flt_scores.mean(), "GC_mean": gc_scores.mean(), "p_value": pval})

df_pd_summary = pd.DataFrame(results_list)
df_pd_summary["p_adj"] = multipletests(df_pd_summary["p_value"], method="fdr_bh")[1]
df_pd_summary = df_pd_summary.sort_values("p_adj")
print(df_pd_summary.to_string())

                       cell_type  FLT_mean   GC_mean       p_value         p_adj
1                B cells (naive)  0.174759  0.019573  7.057275e-11  1.552600e-09
19                   Macrophages -0.277368 -0.426128  9.720278e-06  1.069231e-04
8                        T cells  0.491416  0.432613  4.030124e-03  2.955424e-02
4               HSCs/Progenitors  0.118120  0.069676  8.749583e-03  4.812271e-02
6      B cells (atypical/memory) -0.335111 -0.362057  1.362151e-02  5.993463e-02
14                T cells (CD8+)  0.124108  0.045118  4.042771e-02  1.482350e-01
5                        B cells  0.707915  0.690896  5.733446e-02  1.801940e-01
17    Neutrophils (inflammatory) -1.375072 -1.652160  7.420306e-02  1.907684e-01
20                Innate B cells  0.303646  0.232037  7.804161e-02  1.907684e-01
15                  Plasma cells  0.228411  0.168248  1.065248e-01  2.343545e-01
10               Plasmacytoid DC  0.355163  0.333159  3.000299e-01  5.500549e-01
2                      Monoc

In [36]:
mac_df = adata.obs[adata.obs["cell_type"] == "Macrophages"].copy()
mac_df["condition_GC"] = (mac_df["condition"] == "GC").astype(int)

model_pd_mac = smf.glm("PD_norm_score ~ condition_GC + Q('percent.hb') + Q('nCount_RNA')", data=mac_df).fit()
print(model_pd_mac.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:          PD_norm_score   No. Observations:                  545
Model:                            GLM   Df Residuals:                      541
Model Family:                Gaussian   Df Model:                            3
Link Function:               Identity   Scale:                        0.070661
Method:                          IRLS   Log-Likelihood:                -49.225
Date:                Sun, 20 Sep 2026   Deviance:                       38.227
Time:                        20:01:38   Pearson chi2:                     38.2
No. Iterations:                     3   Pseudo R-squ. (CS):             0.8085
Covariance Type:            nonrobust                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept          -0.0560      0.034     

In [37]:
bnaive_df = adata.obs[adata.obs["cell_type"] == "B cells (naive)"].copy()
bnaive_df["condition_GC"] = (bnaive_df["condition"] == "GC").astype(int)

model_pd_bnaive = smf.glm("PD_norm_score ~ condition_GC + Q('percent.hb') + Q('nCount_RNA')", data=bnaive_df).fit()
print(model_pd_bnaive.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:          PD_norm_score   No. Observations:                 1907
Model:                            GLM   Df Residuals:                     1903
Model Family:                Gaussian   Df Model:                            3
Link Function:               Identity   Scale:                        0.091560
Method:                          IRLS   Log-Likelihood:                -424.33
Date:                Sun, 20 Sep 2026   Deviance:                       174.24
Time:                        20:01:38   Pearson chi2:                     174.
No. Iterations:                     3   Pseudo R-squ. (CS):             0.8496
Covariance Type:            nonrobust                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept           0.6137      0.016     

In [41]:
top5_tables = {}

for trait in ["Alzheimers", "Autoimmune_disease", "Multiple_sclerosis", "PD"]:
    trait_df = summary[summary["trait"] == trait].sort_values("raw_padj").head(5)
    top5_tables[trait] = trait_df[["cell_type", "raw_padj", "raw_direction"]].reset_index(drop=True)
    print(f"\n Top 5: {trait} (before contamination) =")
    print(top5_tables[trait].to_string(index=False))


 Top 5: Alzheimers (before contamination) =
                cell_type     raw_padj raw_direction
          B cells (naive) 4.561956e-13        FLT>GC
              Macrophages 2.763892e-10        FLT>GC
B cells (atypical/memory) 3.672070e-03        GC>FLT
                  B cells 5.095004e-02        GC>FLT
         HSCs/Progenitors 6.687811e-02        FLT>GC

 Top 5: Autoimmune_disease (before contamination) =
      cell_type     raw_padj raw_direction
B cells (naive) 3.523832e-11        GC>FLT
    Macrophages 6.025167e-04        GC>FLT
        B cells 2.525768e-03        GC>FLT
Dendritic cells 1.420328e-01        FLT>GC
 Innate B cells 1.420328e-01        GC>FLT

 Top 5: Multiple_sclerosis (before contamination) =
                cell_type     raw_padj raw_direction
                  B cells 3.318749e-13        GC>FLT
              Macrophages 2.283643e-10        FLT>GC
          B cells (naive) 5.813052e-09        FLT>GC
B cells (atypical/memory) 9.699380e-07        GC>FLT
        

In [47]:
GS_FILE_FULL = "/Users/lexigover/Desktop/OSD-404/GLDS404_STARsolo/gs_file/magma_10kb_top1000_zscore.74_traits.rv1.gs"

cardio_traits = {
    "Coronary_Artery_Disease": "PASS_Coronary_Artery_Disease",
    "Atrial_Fibrillation": "PASS_AtrialFibrillation_Nielsen2018",
    "Cardiovascular_Disease": "UKB_460K.disease_CARDIOVASCULAR",
}

df_gs_cardio = scdrs.util.load_gs(
    GS_FILE_FULL,
    src_species="human",
    dst_species="mouse",
    to_intersect=adata.var_names,
)

for label, trait in cardio_traits.items():
    gene_list = df_gs_cardio[trait][0]
    gene_weight = df_gs_cardio[trait][1]
    print(f"{label}: {len(gene_list)} genes after mouse conversion")

    df_res = scdrs.score_cell(adata, gene_list, gene_weight=gene_weight, n_ctrl=100)
    results[label] = df_res
    adata.obs[f"{label}_norm_score"] = df_res["norm_score"].values
    print(f"{label} done — {df_res.shape[0]} cells scored\n")

Coronary_Artery_Disease: 723 genes after mouse conversion


Computing control scores: 100%|██████████████| 100/100 [00:02<00:00, 34.09it/s]


Coronary_Artery_Disease done — 25294 cells scored

Atrial_Fibrillation: 744 genes after mouse conversion


Computing control scores: 100%|██████████████| 100/100 [00:02<00:00, 33.80it/s]


Atrial_Fibrillation done — 25294 cells scored

Cardiovascular_Disease: 745 genes after mouse conversion


Computing control scores: 100%|██████████████| 100/100 [00:02<00:00, 33.69it/s]


Cardiovascular_Disease done — 25294 cells scored



In [48]:
cardio_raw_rows = []
for trait in ["Coronary_Artery_Disease", "Atrial_Fibrillation", "Cardiovascular_Disease"]:
    score_col = f"{trait}_norm_score"
    for ct in adata.obs["cell_type"].unique():
        sub = adata.obs[adata.obs["cell_type"] == ct]
        flt = sub[sub["condition"] == "FLT"][score_col]
        gc = sub[sub["condition"] == "GC"][score_col]
        if len(flt) > 0 and len(gc) > 0:
            _, pval = stats.mannwhitneyu(flt, gc)
            cardio_raw_rows.append({
                "trait": trait,
                "cell_type": ct,
                "raw_p": pval,
                "raw_direction": "FLT>GC" if flt.mean() > gc.mean() else "GC>FLT"
            })

cardio_raw_df = pd.DataFrame(cardio_raw_rows)
# BH correction within each trait separately
cardio_raw_df["raw_padj"] = cardio_raw_df.groupby("trait")["raw_p"].transform(
    lambda p: multipletests(p, method="fdr_bh")[1]
)

# append to the existing summary data so top5 pulls work the same way
summary = pd.concat([summary, cardio_raw_df[["trait", "cell_type", "raw_padj", "raw_direction"]]], ignore_index=True)

In [55]:
for trait in ["Alzheimers", "Autoimmune_disease", "Multiple_sclerosis", "PD",
              "Coronary_Artery_Disease", "Atrial_Fibrillation", "Cardiovascular_Disease"]:
    trait_df = summary[summary["trait"] == trait].sort_values("raw_padj").head(5)
    top5_tables[trait] = trait_df[["cell_type", "raw_padj", "raw_direction"]].reset_index(drop=True)
    print(f"\nTop 5: {trait} (before contamination)")
    print(top5_tables[trait].to_string(index=False))


Top 5: Alzheimers (before contamination)
                cell_type     raw_padj raw_direction
          B cells (naive) 4.561956e-13        FLT>GC
              Macrophages 2.763892e-10        FLT>GC
B cells (atypical/memory) 3.672070e-03        GC>FLT
                  B cells 5.095004e-02        GC>FLT
         HSCs/Progenitors 6.687811e-02        FLT>GC

Top 5: Autoimmune_disease (before contamination)
      cell_type     raw_padj raw_direction
B cells (naive) 3.523832e-11        GC>FLT
    Macrophages 6.025167e-04        GC>FLT
        B cells 2.525768e-03        GC>FLT
Dendritic cells 1.420328e-01        FLT>GC
 Innate B cells 1.420328e-01        GC>FLT

Top 5: Multiple_sclerosis (before contamination)
                cell_type     raw_padj raw_direction
                  B cells 3.318749e-13        GC>FLT
              Macrophages 2.283643e-10        FLT>GC
          B cells (naive) 5.813052e-09        FLT>GC
B cells (atypical/memory) 9.699380e-07        GC>FLT
                 

In [52]:
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf

traits = ["Alzheimers", "Autoimmune_disease", "Multiple_sclerosis", "PD",
          "Coronary_Artery_Disease", "Atrial_Fibrillation", "Cardiovascular_Disease"]
all_rows = []

for trait in traits:
    score_col = f"{trait}_norm_score"

    # raw FLT vs GC per cell type
    raw_rows = []
    for ct in adata.obs["cell_type"].unique():
        sub = adata.obs[adata.obs["cell_type"] == ct]
        flt = sub[sub["condition"] == "FLT"][score_col]
        gc = sub[sub["condition"] == "GC"][score_col]
        if len(flt) > 0 and len(gc) > 0:
            _, pval = stats.mannwhitneyu(flt, gc)
            raw_rows.append({"cell_type": ct, "FLT_mean": flt.mean(), "GC_mean": gc.mean(), "raw_p": pval})
    raw_df = pd.DataFrame(raw_rows)
    raw_df["raw_padj"] = multipletests(raw_df["raw_p"], method="fdr_bh")[1]

    # GLM-adjusted test per cell type (condition + percent.hb + nCount_RNA)
    for _, row in raw_df.iterrows():
        ct = row["cell_type"]
        ct_df = adata.obs[adata.obs["cell_type"] == ct].copy()
        ct_df["condition_GC"] = (ct_df["condition"] == "GC").astype(int)
        try:
            model = smf.glm(f"{score_col} ~ condition_GC + Q('percent.hb') + Q('nCount_RNA')", data=ct_df).fit()
            glm_p = model.pvalues["condition_GC"]
            glm_coef = model.params["condition_GC"]
        except Exception:
            glm_p, glm_coef = None, None

        all_rows.append({
            "trait": trait,
            "cell_type": ct,
            "raw_padj": row["raw_padj"],
            "raw_direction": "FLT>GC" if row["FLT_mean"] > row["GC_mean"] else "GC>FLT",
            "glm_condition_p": glm_p,
            "glm_direction": ("FLT>GC" if glm_coef < 0 else "GC>FLT") if glm_coef is not None else None,
            "held_up": (glm_p is not None and glm_p < 0.05)
        })

summary = pd.DataFrame(all_rows).sort_values(["trait", "raw_padj"])
pd.set_option('display.max_rows', None)
print(summary.to_string())

                       trait                     cell_type      raw_padj raw_direction  glm_condition_p glm_direction  held_up
1                 Alzheimers               B cells (naive)  4.561956e-13        FLT>GC     8.158721e-02        FLT>GC    False
19                Alzheimers                   Macrophages  2.763892e-10        FLT>GC     9.698618e-02        FLT>GC    False
6                 Alzheimers     B cells (atypical/memory)  3.672070e-03        GC>FLT     6.059604e-03        GC>FLT     True
5                 Alzheimers                       B cells  5.095004e-02        GC>FLT     7.023081e-06        GC>FLT     True
4                 Alzheimers              HSCs/Progenitors  6.687811e-02        FLT>GC     3.462966e-01        FLT>GC    False
14                Alzheimers                T cells (CD8+)  8.753026e-02        FLT>GC     1.960435e-01        GC>FLT    False
3                 Alzheimers                      NK cells  1.795168e-01        GC>FLT     5.270058e-03        

In [53]:
comparison_rows = []

traits = ["Alzheimers", "Autoimmune_disease", "Multiple_sclerosis", "PD",
          "Coronary_Artery_Disease", "Atrial_Fibrillation", "Cardiovascular_Disease"]

for trait in traits:
    score_col = f"{trait}_norm_score"
    for ct in adata.obs["cell_type"].unique():
        ct_df = adata.obs[adata.obs["cell_type"] == ct].copy()
        ct_df["condition_GC"] = (ct_df["condition"] == "GC").astype(int)
        if len(ct_df) == 0:
            continue
        try:
            # contamination only - matches your original Snca/IFN standard
            model_hb_only = smf.glm(f"{score_col} ~ condition_GC + Q('percent.hb')", data=ct_df).fit()
            hb_only_p = model_hb_only.pvalues["condition_GC"]
            hb_only_coef = model_hb_only.params["condition_GC"]

            # contamination + depth - your current scDRS standard
            model_both = smf.glm(f"{score_col} ~ condition_GC + Q('percent.hb') + Q('nCount_RNA')", data=ct_df).fit()
            both_p = model_both.pvalues["condition_GC"]
            both_coef = model_both.params["condition_GC"]
        except Exception:
            continue

        comparison_rows.append({
            "trait": trait,
            "cell_type": ct,
            "hb_only_p": hb_only_p,
            "hb_only_direction": "FLT>GC" if hb_only_coef < 0 else "GC>FLT",
            "hb_only_sig": hb_only_p < 0.05,
            "hb_and_depth_p": both_p,
            "hb_and_depth_direction": "FLT>GC" if both_coef < 0 else "GC>FLT",
            "hb_and_depth_sig": both_p < 0.05,
            "conclusion_changed": (hb_only_p < 0.05) != (both_p < 0.05)
        })

comparison_df = pd.DataFrame(comparison_rows)
pd.set_option('display.max_rows', None)
print(comparison_df.sort_values(["trait", "hb_only_p"]).to_string())

                       trait                     cell_type     hb_only_p hb_only_direction  hb_only_sig  hb_and_depth_p hb_and_depth_direction  hb_and_depth_sig  conclusion_changed
1                 Alzheimers               B cells (naive)  3.669173e-16            FLT>GC         True    8.158721e-02                 FLT>GC             False                True
5                 Alzheimers                       B cells  3.176832e-04            GC>FLT         True    7.023081e-06                 GC>FLT              True               False
19                Alzheimers                   Macrophages  5.159388e-04            FLT>GC         True    9.698618e-02                 FLT>GC             False                True
14                Alzheimers                T cells (CD8+)  1.348926e-02            FLT>GC         True    1.960435e-01                 GC>FLT             False                True
3                 Alzheimers                      NK cells  1.558044e-02            GC>FLT     

In [54]:
print(comparison_df[comparison_df["conclusion_changed"]].to_string())

                      trait                  cell_type     hb_only_p hb_only_direction  hb_only_sig  hb_and_depth_p hb_and_depth_direction  hb_and_depth_sig  conclusion_changed
1                Alzheimers            B cells (naive)  3.669173e-16            FLT>GC         True        0.081587                 FLT>GC             False                True
6                Alzheimers  B cells (atypical/memory)  6.030684e-01            FLT>GC        False        0.006060                 GC>FLT              True                True
8                Alzheimers                    T cells  1.014309e-01            GC>FLT        False        0.002401                 GC>FLT              True                True
9                Alzheimers     Neutrophils (immature)  1.437655e-01            FLT>GC        False        0.006719                 FLT>GC              True                True
13               Alzheimers       B cells (progenitor)  4.631136e-02            FLT>GC         True        0.731815